# Logistic Regression

The classifiers we have seen so far answer with a hard decision.
Logistic regression answers with a **probability**, which is often what a decision maker actually
wants: not "this customer will subscribe", but "this customer subscribes with probability $0.8$".

The model is
$$P(Y = 1 \mid \vec{x}) = \sigma\left( \beta_0 + \vec{w} \cdot \vec{x} \right)
\qquad \text{where} \qquad
\sigma(z) = \frac{1}{1 + e^{-z}}$$

and its parameters are chosen by minimizing the **log loss**.
In this notebook we write that cost function ourselves, hand it to a solver, and then read and
plot what comes out.

## Generated data

Step -1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

Step 0: The sigmoid

Write the function $\sigma(z) = \dfrac{1}{1+e^{-z}}$ and plot it between $-8$ and $8$.

Check on the picture that it takes its values in $(0,1)$, that it is increasing, and that
$\sigma(0) = \frac{1}{2}$: this last point is the one that makes the decision boundary sit at
$\beta_0 + \vec{w} \cdot \vec{x} = 0$.

In [ ]:
# def sigmoid(z):
#     ...

# plot it

Step 1: Generate the data

We invent $200$ students.
For each of them, draw the number of hours they revised, `x`, uniformly between $0$ and $10$
(`rng.uniform`).

Then draw whether they passed.
This is the part that matters: the label is **not** a deterministic function of `x`.
We fix a true model
$$\beta_0^{\text{true}} = -3, \qquad \beta_1^{\text{true}} = 0.8$$
and, for each student, we say that they passed with probability $\sigma(\beta_0 + \beta_1 x)$.
In code, `y = (rng.random(n) < p).astype(float)` does exactly that: it draws a uniform number in
$(0,1)$ and compares it to the probability.

The labels must be $0$ and $1$ here (and not $\pm 1$), because that is the convention the log loss
below is written for.

In [ ]:
rng = np.random.default_rng(0)
n = 200
beta_true = np.array([-3.0, 0.8])

# x = ...
# p_true = ...
# y = ...

# y.sum(), n   # how many students passed?

Step 2: Plot the data

Draw the students as points at height $0$ or $1$ (a vertical tick, `marker="|"`, is more readable
than a dot when many points pile up), with one colour per class, and add the curve of the true
model on top.

Look at the middle of the picture: around $4$ hours, both outcomes occur.
No decision boundary can separate these points, and that is not a defect of the data, it is what
we are trying to model: in that region the probability of passing is genuinely close to
$\frac{1}{2}$.

## The problem to solve

We look for the $\beta_0$ and $\beta_1$ that make the observed labels as likely as possible.
Writing $p_k = \sigma\left( \beta_0 + \beta_1 x_k \right)$, the quantity to minimize is the
**log loss**

$$\mathcal{L}(\beta_0, \beta_1)
= - \sum_{k=1}^{n} \Big[ y_k \log\left( p_k \right) + \left( 1 - y_k \right) \log\left( 1 - p_k \right) \Big]$$

Since $y_k$ is $0$ or $1$, only one of the two terms survives for each student, and the cost of a
student is $-\log$ of the probability the model gave to the **correct** answer.

For the code, it is worth rewriting each term.
Write $z_k = \beta_0 + \beta_1 x_k$, so that $p_k = \sigma(z_k)$ and, since
$1 - \sigma(z) = \sigma(-z)$, the cost of one student is

$$\ell_k = - y_k \log \sigma(z_k) - \left( 1 - y_k \right) \log \sigma(-z_k)$$

Both logarithms can be opened up with

$$\log \sigma(z) = \log \frac{1}{1+e^{-z}} = -\log\left( 1 + e^{-z} \right)$$

which turns the two terms into

$$\ell_k = y_k \log\left( 1 + e^{-z_k} \right) + \left( 1 - y_k \right) \log\left( 1 + e^{z_k} \right)$$

that is, reading the two cases separately,

$$\ell_k = \log\left( 1 + e^{-z_k} \right) \ \text{ if } y_k = 1,
\qquad \qquad
\ell_k = \log\left( 1 + e^{z_k} \right) \ \text{ if } y_k = 0$$

The two cases can be merged into a single expression, which is what we want for the code.
Factor $e^{z}$ out of the first logarithm:

$$\log\left( 1 + e^{-z} \right)
= \log\left( e^{-z} \left( e^{z} + 1 \right) \right)
= \log\left( 1 + e^{z} \right) - z$$

The case $y_k = 1$ therefore reads $\log\left( 1 + e^{z_k} \right) - z_k$, and the case
$y_k = 0$ reads $\log\left( 1 + e^{z_k} \right)$.
They differ only by the term $-z_k$, which is present exactly when $y_k = 1$, so both are covered
by the single expression

$$\boxed{ \ \ell_k = \log\left( 1 + e^{z_k} \right) - y_k z_k \ }$$

which is the form to program.
The total cost is $\mathcal{L} = \sum_k \ell_k$.

**Why not simply code the first formula?**
Because it computes $p_k$ first, and $p_k$ saturates.
In double precision, $\sigma(50)$ is rounded to exactly $1.0$, so $1 - p_k$ becomes $0.0$ and
$\log(1-p_k)$ returns $-\infty$, although the true cost of that point is a perfectly finite $50$.
The same happens on the other side: $e^{-z}$ overflows to infinity as soon as $z < -710$ or so,
and $p_k$ collapses to $0.0$.
A single such point makes the whole cost infinite, the gradient meaningless, and the solver stops
on a wall that does not exist.

The boxed form never forms $p_k$ at all, and `np.logaddexp(0, z)` evaluates $\log(1+e^z)$ safely
for every $z$ (internally it computes $\max(0,z) + \log(1 + e^{-|z|})$, where the exponential is
always of a negative number, hence between $0$ and $1$).
For $z = 50$ it simply returns $50.0$, as it should.

There is no formula for the minimum, so we give the function to a solver.
It converges much faster if we also give it the gradient, which the notes derive:

$$\frac{\partial \mathcal{L}}{\partial \beta_0} = \sum_k \left( p_k - y_k \right)
\qquad \text{and} \qquad
\frac{\partial \mathcal{L}}{\partial \beta_1} = \sum_k x_k \left( p_k - y_k \right)$$

Step 3: Write the cost and its gradient

In [ ]:
# def loss(beta):
#     z = ...
#     return ...

# def loss_grad(beta):
#     p = ...
#     return np.array([..., ...])

# loss(np.array([0.0, 0.0]))   # every p is 1/2, so this should be n*log(2)

Step 4: Minimize it

`scipy.optimize.minimize` with the method `BFGS` is the natural choice here: the cost is smooth
and there is no constraint at all (any $\beta_0$ and $\beta_1$ are allowed).

Start from $\beta = (0,0)$, pass the gradient through `jac=`, and check `res.success`.

In [ ]:
# res = minimize(...)
# res.success, res.x, res.fun

Step 5: Check the answer

Two checks are worth doing.

1. Compare the fitted parameters with the ones we used to generate the data. They will not be
   equal: we only saw $200$ noisy students, not the model itself.
2. At the minimum the gradient vanishes, which gives the two identities
   $$\sum_k p_k = \sum_k y_k \qquad \text{and} \qquad \sum_k x_k \, p_k = \sum_k x_k \, y_k$$
   These hold exactly at the optimum, whatever the data, and they are the cheapest way of
   checking that a solver has really converged.

Step 6: Plot the fitted model

Same picture as in Step 2, with the fitted curve added, and a vertical line at the decision
boundary, that is where $\beta_0 + \beta_1 x = 0$.

Step 7: Read the fitted model

Answer three questions:

- what probability of passing does the model give to a student who revised $2$ hours? $5$ hours?
  $8$ hours?
- above how many hours does it predict a pass?
- by what factor does one extra hour of revision multiply the **odds** of passing?

For the last one, remember that the model is linear in the log-odds:
$\log \frac{p}{1-p} = \beta_0 + \beta_1 x$, so adding one hour adds $\beta_1$ to the log-odds,
hence multiplies the odds by $e^{\beta_1}$.

Step 8: How much can we trust these numbers?

Refit the model using only the first $30$ students, then only the first $60$, and compare with the
fit on all $200$.

The estimate moves a lot: a logistic regression fitted on a handful of noisy points is not worth
much, and nothing in the output of the solver warns you about it.

---

## Loan applications

A bank has kept the record of $60$ past loan applications that were all accepted.
For each one it knows:

| column | meaning |
|---|---|
| `income` | annual income of the applicant, in thousands of euros |
| `years_employed` | number of years in the current job |
| `repaid` | $+1$ if the loan was repaid, $-1$ if the applicant defaulted |

This time the bank does not want a yes or no: it wants the **probability** that a new applicant
repays, so that it can decide where to put its threshold itself.

The data is in `loans.csv`, next to this notebook
(or from GitHub: https://github.com/pauldubois98/RefresherMaths2026/blob/main/SessionBinaryClassification/loans.csv).

Step 0: Read the data

Careful with the labels: the file stores $\pm 1$, and the log loss above is written for labels
$0$ and $1$. Convert them.

In [ ]:
import pandas as pd

df = pd.read_csv('loans.csv')
df.head()

In [ ]:
# X_loans = ...            # shape (60, 2)
# y_loans = ...            # 0 and 1, not -1 and +1

# X_loans.shape, y_loans[:10]

Step 1: Plot the two classes

Step 2: Standardize the features

Subtract the mean and divide by the standard deviation of each column, keeping `mu` and `sigma`
aside: we will need them to translate the fitted coefficients back into euros and years.

Why bother?
One column is measured in thousands of euros and runs up to $90$, the other is a number of years
and runs up to $20$.
Nothing in the log loss forbids that, but the two coefficients would then live on wildly
different scales, the solver would zigzag, and the penalty we are about to add in Step 5 (which
treats all the coefficients alike) would be meaningless.

In [ ]:
# mu = ...
# sigma = ...
# Z = ...

Step 3: The cost function, with two features

The model is now
$$P(\text{repaid} \mid \vec{x}) = \sigma\left( \beta_0 + w_1 x_1 + w_2 x_2 \right)$$

Rather than carrying $\beta_0$ separately, add a column of ones to the data
(`np.hstack([np.ones((len(Z), 1)), Z])`).
The three unknowns are then a single vector $\theta = (\beta_0, w_1, w_2)$, the model reads
$z = A \theta$ where $A$ is that matrix, and the cost and its gradient become

$$\mathcal{L}(\theta) = \sum_k \left[ \log\left( 1 + e^{z_k} \right) - y_k z_k \right]
\qquad \qquad
\nabla \mathcal{L}(\theta) = A^T \left( \vec{p} - \vec{y} \right)$$

which is the same thing as before, written for any number of features.
Write `A`, the cost and the gradient.

In [ ]:
# A = ...

# def loss2(theta):
#     ...

# def loss2_grad(theta):
#     ...

# loss2(np.zeros(3))

Step 4: Minimize it, and look carefully at the result

In [ ]:
# res2 = minimize(...)
# print the parameters and the value of the cost

Step 5: Something is wrong

The solver reports a success, but the coefficients are enormous and the cost is essentially zero:
the model claims to be certain about all $60$ applicants.

To see what is happening, run the solver again with a limit on the number of iterations,
`options={"maxiter": k}` for `k` equal to $5$, $10$, $20$, $50$ and $100$, and print the norm of
$\theta$ and the cost each time.

The coefficients grow without ever settling, and the cost keeps decreasing towards $0$.

The reason is in the picture of Step 1: the two classes are **perfectly separable**, there is a
straight line with all the repaid applicants on one side and all the defaults on the other.
Take any such line, multiply $\theta$ by $10$: the boundary does not move, but every $\sigma$
gets closer to $0$ or $1$, so every prediction becomes more confident and the loss gets smaller.
There is always a better $\theta$, so the minimum does not exist, and what stops the solver in the
end is only the arithmetic of the machine.

Note the contrast with Part 1, where the two classes overlapped: **overlap is what gives logistic
regression a finite answer**.

Step 6: The usual fix, a penalty

We add to the cost a term that grows with the size of the coefficients:

$$\mathcal{L}_\lambda(\theta) = \mathcal{L}(\theta) + \lambda \left( w_1^2 + w_2^2 \right)$$

Now blowing $\theta$ up has a price, and the minimum exists.
The intercept $\beta_0$ is left out of the penalty: it only sets where the boundary is, not how
sharp the model is, and penalizing it would pull the predicted probabilities towards
$\frac{1}{2}$ for no good reason.

Write the penalized cost and its gradient (the extra term contributes $2\lambda w$ to the
gradient, and nothing to the derivative in $\beta_0$), and minimize with $\lambda = 1$.

In [ ]:
# def loss2_pen(theta, lam=1.0):
#     ...

# def loss2_pen_grad(theta, lam=1.0):
#     ...

# res3 = minimize(...)
# theta = res3.x

Step 7: Plot the fitted model

Draw the two classes again, and on top of them the probability given by the model.

A convenient way is to build a grid over the picture with `np.meshgrid`, evaluate the model at
every point of the grid, and use `plt.contourf` for the colours and `plt.contour` for a few level
lines, for instance $0.25$, $0.5$ and $0.75$.
The line $P = 0.5$ is the decision boundary; the other two show how quickly the model changes its
mind, which is the whole point of predicting a probability rather than a class.

Careful: the model was fitted on the standardized data, so a point of the grid has to be
standardized before being fed to it.

Step 8: Read the coefficients

The model was fitted on standardized features, so its coefficients are not directly readable.
Translate them back:

$$\beta_0 + \vec{w} \cdot \frac{\vec{x} - \mu}{\sigma}
= \underbrace{\left( \beta_0 - \sum_i \frac{w_i \mu_i}{\sigma_i} \right)}_{b_\text{orig}}
+ \underbrace{\left( \frac{w_1}{\sigma_1}, \frac{w_2}{\sigma_2} \right)}_{\vec{w}_\text{orig}} \cdot \vec{x}$$

Then give the two numbers the bank can actually use: by what factor does the probability, in
odds, get multiplied by one more thousand euros of income, and by one more year of employment?

Step 9: Three new applicants

| | income | years employed |
|---|---|---|
| A | $40$ | $5$ |
| B | $55$ | $8$ |
| C | $30$ | $18$ |

Give the probability of repayment for each of them, and not only the decision.
Which of the three is the bank still in the dark about?

Step 10: Compare with a library

`sklearn` fits exactly the same penalized model, but writes it as

$$C \, \mathcal{L}(\theta) + \frac{1}{2} \|\vec{w}\|^2$$

Dividing by $C$, this is our cost with $\lambda = \dfrac{1}{2C}$, so our $\lambda = 1$
corresponds to $C = 0.5$.

Fit `LogisticRegression(C=0.5)` on the standardized data and compare `intercept_` and `coef_`
with your $\theta$. Compare also `predict_proba` with the probabilities of Step 9.

In [ ]:
# from sklearn.linear_model import LogisticRegression

Step 11: To finish

Three questions worth thinking about.

1. Refit with $\lambda = 0.01$ and with $\lambda = 100$, and look at the picture of Step 7 each
   time. What is the penalty really controlling?
2. The bank must eventually say yes or no, so it needs a threshold. Nothing forces it to be
   $\frac{1}{2}$: refusing a good client and accepting a bad one do not cost the same. Where, in
   the plot of Step 7, would a threshold of $0.8$ put the boundary?
3. Our two features enter only through $\beta_0 + w_1 x_1 + w_2 x_2$, so the level lines of the
   probability are parallel straight lines. What would you add to the model to allow a curved
   boundary, without changing anything to the code that minimizes the cost?